# Chapitre 4 - Exercices 4.6 à 4.9

- **4.6** : convergence du perceptron ;
- **4.7** : un critère « tous points » pour les hyperplans séparateurs ;
- **4.8** : modèle gaussien à centroïdes de rang réduit et LDA de rang réduit ;
- **4.9** : programmer la QDA et l'appliquer aux voyelles.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import linalg, optimize
from esl.plotting import set_style, COLORS, BLUE, ORANGE, GRAY, SEQ, DIV, TWO_CLASS
from esl import datasets
from esl import ch04_classification as c4
set_style()

## Exercice 4.6 : Convergence du perceptron

**Énoncé.** On dispose de $N$ points $x_i\in\mathbb{R}^p$ en position générale, d'étiquettes
$y_i\in\{-1,1\}$, et l'on suppose les classes linéairement séparables. Montrer que
l'algorithme du perceptron converge vers un hyperplan séparateur en un nombre fini
d'étapes :

(a) On note un hyperplan $f(x) = \beta_1^Tx+\beta_0 = 0$, ou $\beta^Tx^* = 0$ avec
$x^* = (x,1)$ et $\beta = (\beta_1,\beta_0)$. Soit $z_i = x_i^*/\|x_i^*\|$. Montrer que la
séparabilité implique l'existence d'un $\beta_{\text{sep}}$ tel que $y_i\beta_{\text{sep}}^Tz_i\ge1$ pour tout $i$.

(b) Étant donné $\beta_{\text{old}}$, le perceptron choisit un point $z_i$ mal classé et
effectue $\beta_{\text{new}}\leftarrow\beta_{\text{old}}+y_iz_i$. Montrer que
$\|\beta_{\text{new}}-\beta_{\text{sep}}\|^2\le\|\beta_{\text{old}}-\beta_{\text{sep}}\|^2-1$,
et en déduire que l'algorithme converge vers un hyperplan séparateur en au plus
$\|\beta_{\text{start}}-\beta_{\text{sep}}\|^2$ étapes (Ripley, 1996).

### (a) Existence de $\beta_{\text{sep}}$

Les classes étant séparables, il existe $\beta = (\beta_1,\beta_0)$ tel que
$y_i(\beta_1^Tx_i+\beta_0) > 0$, c'est-à-dire $y_i\beta^Tx_i^* > 0$, pour tout $i$.

- $\|x_i^*\|\ge1 > 0$ (sa dernière coordonnée vaut 1), donc $z_i$ est bien défini et
  $y_i\beta^Tz_i = y_i\beta^Tx_i^*/\|x_i^*\| > 0$.
- Il y a un nombre **fini** de points, donc $m = \min_iy_i\beta^Tz_i$ est atteint et $m > 0$.

Posons $\beta_{\text{sep}} = \beta/m$. Par linéarité, $y_i\beta_{\text{sep}}^Tz_i = (y_i\beta^Tz_i)/m\ge1$ pour tout $i$. $\square$

### (b) Décroissance de la distance à $\beta_{\text{sep}}$

On développe le carré de la norme ($\|a+b\|^2 = \|a\|^2+2a^Tb+\|b\|^2$) :
$$\|\beta_{\text{new}}-\beta_{\text{sep}}\|^2 = \|\beta_{\text{old}}-\beta_{\text{sep}}\|^2 + 2y_i(\beta_{\text{old}}-\beta_{\text{sep}})^Tz_i + y_i^2\|z_i\|^2.$$
Examinons les deux derniers termes :

- $y_i^2\|z_i\|^2 = 1$ car $y_i = \pm1$ et $\|z_i\| = 1$ ;
- $y_i\beta_{\text{old}}^Tz_i\le0$ car $z_i$ est **mal classé** par $\beta_{\text{old}}$ ;
- $y_i\beta_{\text{sep}}^Tz_i\ge1$ par (a).

D'où $2y_i(\beta_{\text{old}}-\beta_{\text{sep}})^Tz_i\le 2(0-1) = -2$, et
$$\|\beta_{\text{new}}-\beta_{\text{sep}}\|^2\le\|\beta_{\text{old}}-\beta_{\text{sep}}\|^2-2+1 = \|\beta_{\text{old}}-\beta_{\text{sep}}\|^2-1.$$

**Conclusion.** Après $k$ mises à jour, $0\le\|\beta^{(k)}-\beta_{\text{sep}}\|^2\le\|\beta_{\text{start}}-\beta_{\text{sep}}\|^2-k$,
donc $k\le\|\beta_{\text{start}}-\beta_{\text{sep}}\|^2$ : le nombre de mises à jour est
fini. Quand l'algorithme s'arrête, plus aucun point n'est mal classé, c'est-à-dire
$y_i\beta^Tz_i > 0$ pour tout $i$ - et comme $\beta^Tz_i$ a le signe de $\beta^Tx_i^*$,
l'hyperplan $\beta^Tx^* = 0$ **sépare** les données. $\square$

> **Remarques.**
> - La borne vaut pour **tout** $\beta_{\text{sep}}$ admissible ; la meilleure s'obtient
>   avec celui de norme minimale. En partant de $\beta_{\text{start}} = 0$, elle vaut
>   $\|\beta_{\text{sep}}\|^2 = 1/\gamma^2$, où
>   $\gamma = \max_{\|\beta\|=1}\min_iy_i\beta^Tz_i$ est la « marge » des $z_i$ : plus les
>   classes sont proches, plus la borne est grande (troisième problème du perceptron cité
>   dans le livre).
> - L'ordre de visite des points mal classés est quelconque : la preuve ne l'utilise pas.

### Vérification numérique

On calcule le $\beta_{\text{sep}}$ de norme minimale, solution de
$\min\|\beta\|^2$ sous $y_i\beta^Tz_i\ge1$ (programmation quadratique avec SLSQP), puis on
fait tourner le perceptron normalisé (fonction `perceptron(..., normalize=True)`) avec
différents ordres de parcours.

In [2]:
rng = np.random.RandomState(3)
Xp = np.vstack([rng.randn(25, 2) * 0.6 + [-1.0, 0.5], rng.randn(25, 2) * 0.6 + [1.0, -0.3]])
yp = np.repeat([-1, 1], 25)
r0 = c4.perceptron(Xp, yp, max_epochs=10000)
print("Données séparables :", r0["converged"])
Z = np.column_stack([Xp, np.ones(len(yp))])
Z /= np.linalg.norm(Z, axis=1, keepdims=True)
A = yp[:, None] * Z
start = r0["beta"] / np.min(A @ r0["beta"])                 # point admissible
sol = optimize.minimize(lambda b: b @ b, start, jac=lambda b: 2 * b, method="SLSQP",
                        constraints=[{"type": "ineq", "fun": lambda b: A @ b - 1, "jac": lambda b: A}],
                        options={"ftol": 1e-12, "maxiter": 500})
beta_sep = sol.x
borne = beta_sep @ beta_sep
print("min_i y_i beta_sep^T z_i = %.4f ;  borne ||beta_sep||^2 = %.1f" % (np.min(A @ beta_sep), borne))

Données séparables : True
min_i y_i beta_sep^T z_i = 1.0000 ;  borne ||beta_sep||^2 = 556.4


In [3]:
nb = [c4.perceptron(Xp, yp, normalize=True, rng=np.random.RandomState(s))["n_updates"] for s in range(200)]
print("Mises à jour sur 200 ordres de parcours : min %d, médiane %d, max %d (borne %.0f)" % (
    min(nb), np.median(nb), max(nb), borne))

Mises à jour sur 200 ordres de parcours : min 2, médiane 29, max 67 (borne 556)


In [ ]:
r = c4.perceptron(Xp, yp, normalize=True, rng=np.random.RandomState(0))
dist2 = np.sum((r["path"] - beta_sep) ** 2, axis=1)
k = np.arange(len(dist2))
fig, ax = plt.subplots(figsize=(7, 4.2))
ax.plot(k, dist2, "o-", ms=3, color=BLUE, label=r"$\|\beta^{(k)}-\beta_{\mathrm{sep}}\|^2$")
ax.plot(k, dist2[0] - k, "--", color=ORANGE, label=r"borne $\|\beta^{(0)}-\beta_{\mathrm{sep}}\|^2 - k$")
ax.set_title("Exercice 4.6 : chaque mise à jour rapproche de $\\beta_{\\mathrm{sep}}$")
ax.set_xlabel("nombre de mises à jour $k$")
ax.set_ylabel("distance au carré")
ax.legend()
plt.show()
print("Décroissance d'au moins 1 à chaque pas :", np.all(np.diff(dist2) <= -1 + 1e-9))

La distance au carré décroît d'au moins 1 à chaque mise à jour, et le nombre de mises à
jour reste toujours sous la borne (souvent très en dessous : la borne est pessimiste).

> **À retenir.** Pour des données séparables, chaque correction du perceptron (sur les
> points normalisés $z_i$) réduit d'au moins 1 le carré de la distance à un séparateur
> $\beta_{\text{sep}}$ fixé : l'algorithme s'arrête après au plus
> $\|\beta_{\text{start}}-\beta_{\text{sep}}\|^2$ mises à jour, sur un hyperplan séparant.

## Exercice 4.7 : Un critère sur tous les points

**Énoncé.** On considère le critère
$$D^*(\beta,\beta_0) = -\sum_{i=1}^Ny_i(x_i^T\beta+\beta_0)\qquad\text{(éq. (4.58))},$$
généralisation de (4.41) où l'on somme sur **toutes** les observations (et pas seulement
sur les mal classées). On minimise $D^*$ sous la contrainte $\|\beta\| = 1$. Décrire ce
critère en mots. Résout-il le problème de l'hyperplan séparateur optimal ?